# Demo: MIL-GP with weighted Noisy-OR

This notebook trains the GP-based MIL model on one torchmil dataset, evaluates it, and shows the
instance-level probabilities, weights and uncertainties for one positive bag.

It runs on Google Colab (GPU recommended) or locally. RSNA is used by default because it is the
smallest download (~280 MB).

## 1. Setup

In [ ]:
# On Colab: clone the repository first (replace with your repository URL)
# !git clone https://github.com/<user>/mil-gp-wnor.git
# %cd mil-gp-wnor
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torchmil.data import collate_fn

from mil_gp import CONFIGS, MILGP, MILTrainer, load_datasets, make_loaders, prepare_dataset, run_experiments

DATASET = "rsna"        # 'rsna', 'panda' or 'camelyon16'
DATA_DIR = "/content"   # where the compressed files are stored
EPOCHS = 20             # short run for the demo (paper settings are in CONFIGS)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CONFIGS[DATASET]
cfg

## 2. Data
Download, extract and split the data (train / stratified validation / test).

In [ ]:
root = prepare_dataset(DATASET, data_dir=DATA_DIR)
train, val, test = load_datasets(DATASET, root, load_at_init=cfg["load_at_init"])
train_loader, val_loader, test_loader = make_loaders(train, val, test, batch_size=cfg["batch_size"])

## 3. Training and evaluation
One model, one seed. The best epoch is selected by validation AUC.

In [ ]:
train_cfg = {**cfg["train"], "epochs": EPOCHS, "seeds": (0,)}
results, history, tables = run_experiments(
    train_loader, val_loader, test_loader,
    save_dir=f"results/demo_{DATASET}",
    model_config=cfg["model"], device=DEVICE, **train_cfg,
)
results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history["epoch"], history["train_loss"]); ax[0].set_title("Train -ELBO")
ax[1].plot(history["epoch"], history["val_auc"]); ax[1].set_title("Validation AUC")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 4. Instance-level interpretation

For a positive test bag, the model returns per instance:
- `E_p ± std`: probability that the instance is positive and its uncertainty,
- `E_w ± std`: weight of the instance in the Noisy-OR aggregation.

In [ ]:
# Reload the best checkpoint of the demo run
ckpt = torch.load(results["checkpoint"].iloc[0], map_location=DEVICE, weights_only=False)
model = MILGP(**ckpt["model_config"])
model.load_state_dict(ckpt["model_state_dict"])
trainer = MILTrainer(model, DEVICE, num_train_bags=1)  # used here only for its forward pass
model.eval()

# First positive bag of the test set
idx = next(i for i in range(len(test)) if test[i]["Y"].item() == 1)
batch = collate_fn([test[idx]])

with torch.no_grad():
    output, Y, mask = trainer._forward(batch)
    y_hat, var_bag, E_p, Var_p, E_w, Var_w = model.likelihood(output)

print(f"Bag {idx}: label={int(Y.item())} | P(Y=1)={y_hat.item():.3f} ± {var_bag.clamp(min=0).sqrt().item():.3f}")

In [ ]:
to_np = lambda t: t[0].cpu().numpy()
p, sp = to_np(E_p), np.sqrt(np.maximum(to_np(Var_p), 0))
w, sw = to_np(E_w), np.sqrt(np.maximum(to_np(Var_w), 0))
y_inst = batch["y_inst"][0].numpy()
x = np.arange(len(p))

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for a in ax:  # shade the truly positive instances
    for i in np.where(y_inst == 1)[0]:
        a.axvspan(i - 0.5, i + 0.5, color="tab:red", alpha=0.15, lw=0)
ax[0].errorbar(x, p, yerr=sp, fmt="o", ms=3, capsize=2); ax[0].set_ylabel("instance probability")
ax[1].errorbar(x, w, yerr=sw, fmt="o", ms=3, capsize=2, color="tab:green"); ax[1].set_ylabel("instance weight")
ax[1].set_xlabel("instance (slice / patch)")
ax[0].set_title("Shaded: ground-truth positive instances")
plt.tight_layout(); plt.show()